In [2]:
import pandas as pd
import os
import glob
from dateutil import parser

# Parameters
ENTRY_TIME = "09:30:00+05:30"
EXIT_TIME = "11:00:00+05:30"
VOL_LOOKBACK_DAYS = 3

In [3]:
def read_stock_data(file):
    df = pd.read_csv(file, parse_dates=["date"])
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)
    df = df.sort_index()
    return df





def calculate_volatility(df, lookback=3):
    # Calculate 5-min returns
    returns = df["close"].pct_change()

    # Extract just the date part for grouping, but keep as pandas Timestamp
    df = df.copy()
    df["date_only"] = df.index.floor("D")

    # Calculate daily std of 5-min returns
    daily_vol = returns.groupby(df["date_only"]).std()

    # Smooth using rolling average
    smoothed_vol = daily_vol.rolling(window=lookback).mean()

    return smoothed_vol



# def predict_direction(df_day):
#     try:
#         row = df_day.between_time("09:30", "09:30").iloc[0]
#         score = 0

#         # === Trend Strength ===
#         if row["ema5"] > row["sma5"]:
#             diff = row["ema5"] - row["sma5"]
#             if diff > 0.2:  # strong trend
#                 score += 1.5
#             else:
#                 score += 1

#         if row["sma5"] > row["sma10"]:
#             diff = row["sma5"] - row["sma10"]
#             score += 1 if diff > 0.1 else 0.5

#         # === Momentum (High Priority) ===
#         if row["macd1226"] > 0:
#             score += 1.5
#         if row["RSI14"] > 60:
#             score += 1.5
#         elif row["RSI14"] > 55:
#             score += 1
#         if row["MOM10"] > 0:
#             score += 1.5 if row["MOM10"] > 0.5 else 1

#         # === Volatility & Breakouts ===
#         if row["close"] > row["upperband"]:
#             score += 1.5
#         if row["KAMA10"] > row["KAMA20"]:
#             diff = row["KAMA10"] - row["KAMA20"]
#             score += 1 if diff > 0.2 else 0.5

#         # === Trend Strength Confirmation ===
#         if row["ADX10"] > 25:
#             score += 1.5
#         elif row["ADX10"] > 20:
#             score += 1

#         # === Final Decision ===
#         if score >= 6:
#             return "buy"
#         else:
#             return "sell"

#     except Exception as e:
#         print(f"Error in predict_direction: {e}")
#         return "sell"




def predict_direction(df_day):
    try:
        row = df_day.between_time("09:30", "09:30").iloc[0]
        if row["RSI14"] < 30:
            return "sell"
        elif row["RSI14"] > 70:
            return "buy"
        else:
            return "hold"
    except:
        return "hold"

def simulate_trade(df_day, direction):
    entry_price = df_day.at_time(ENTRY_TIME)["open"].iloc[0]
    exit_price = df_day.at_time(EXIT_TIME)["open"].iloc[0]
    if direction == "buy":
        return (exit_price - entry_price) / entry_price
    elif direction == "sell":
        return (entry_price - exit_price) /entry_price
    else:
      return 0







def process_day(date, all_stock_data):
    print(f"Processing date: {date}", flush=True)
    volatilities = {}

    for name, df in all_stock_data.items():
        try:
            df_filtered = df[df.index.date <= date]
            if len(df_filtered) < VOL_LOOKBACK_DAYS * 78:  # 78 = 5-min bars/day
                continue
            vol_series = calculate_volatility(df_filtered)
            if not vol_series.empty:
                vol = vol_series.iloc[-1]
                volatilities[name] = vol
        except Exception as e:
            print(f"Error processing {name} on {date}: {e}", flush=True)
            continue

    if not volatilities:
        print(f"No volatility data for {date}", flush=True)
        return None

    most_volatile_stock = max(volatilities, key=volatilities.get)
    print(f"Most volatile: {most_volatile_stock}", flush=True)
    df = all_stock_data[most_volatile_stock]

    try:
        df_day = df[df.index.date == pd.to_datetime(date).date()]
        if df_day.empty:
            return None

        direction = predict_direction(df_day)
        ret = simulate_trade(df_day, direction)

        return {
            "date": date,
            "stock": most_volatile_stock,
            "direction": direction,
            "return": ret
        }
    except Exception as e:
        print(f"Simulation failed on {date} for {most_volatile_stock}: {e}", flush=True)
        return None


In [4]:

files = glob.glob("/content/*_with_indicators_.csv")
all_stock_data = {os.path.basename(f).split(".")[0]: read_stock_data(f) for f in files}
all_dates = sorted(set.union(*[set(df.index.date) for df in all_stock_data.values()]))
start_date = all_dates[0]
end_date = all_dates[-1]

print("Start date:", start_date)
print("End date:", end_date)




Start date: 2015-02-02
End date: 2022-02-18


In [7]:
results = []
capital = 1000
target=1300
leverage = 3
tax_rate=0.003
daily_stoploss = -0.05
print(len(all_dates))#1746

1746


In [8]:
for date in all_dates[760:1590]:
    res = process_day(date, all_stock_data)
    if res:

        daily_return = res["return"] * leverage
        if daily_return < daily_stoploss:
            daily_return = daily_stoploss
        capital *= (1 + daily_return)
        if daily_return != 0:
            capital -= capital * tax_rate
        res["capital"] = capital  # Track capital over time
        res["daily_leveraged_return"] = daily_return
        results.append(res)
      # ✅ Stop if 30% return reached
        if capital >= target:
            print(f"🎯 Target reached on {date}. Capital: ₹{capital:.2f}")
            break



results_df = pd.DataFrame(results)
results_df.to_csv("strategy_results.csv", index=False)
print("Backtest complete. Saved to strategy_results.csv")


Processing date: 2018-02-27
Most volatile: KOTAKBANK_with_indicators_
Processing date: 2018-02-28
Most volatile: KOTAKBANK_with_indicators_
Processing date: 2018-03-01
Most volatile: KOTAKBANK_with_indicators_
Processing date: 2018-03-05
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2018-03-06
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2018-03-07
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2018-03-08
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2018-03-09
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2018-03-12
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2018-03-13
Most volatile: INDIGO_with_indicators_
Processing date: 2018-03-14
Most volatile: INDIGO_with_indicators_
Processing date: 2018-03-15
Most volatile: KOTAKBANK_with_indicators_
Processing date: 2018-03-16
Most volatile: KOTAKBANK_with_indicators_
Processing date: 2018-03-19
Most volatile: KOTAKBANK_with_indicators_
Processing date: 201